# Multiclass classification using the tutorial regression tools
**Workflow:** raw features -> polynomial features; class labels -> one-hot targets; fit one weight column per class; choose the largest test score.

Edit `X`, `y`, `X_test`, `order`, and `lmd`, then **Run All**. Use `order = 1` for linear classification and a higher order for polynomial classification. `lmd = 0` gives ordinary regression; a positive value adds ridge when requested. Inputs and the manual inverse selection follow your polynomial/ridge notebooks.

This notebook uses only libraries/functions already used in Tutorials 3-6 and your existing notebooks. The example data is from Tutorial 6 Q5.


In [28]:
import numpy as np
from numpy.linalg import inv
from sklearn.preprocessing import PolynomialFeatures, OneHotEncoder
from sklearn.metrics import accuracy_score


## 1. Enter training labels and test features
One row of `X` is one training sample. Supply **raw features**, without adding a bias column. Supply **one class label per row** in `y`; the encoder creates the one-hot targets.

The labels may be numbers such as `1, 2, 3` or `10, 20, 30`, or strings. Only classes present in the training labels receive weight columns. `X_test` must have the same raw feature count and ordering as `X`.

Set `y_test = None` when the correct test labels are unknown. Otherwise enter one true label per test sample to calculate test accuracy.


In [29]:
# EDIT INPUTS ? Tutorial 6 Q5 example with three classes.
X = np.array([[1,2,3], 
[4,0,6], 
[1,1,0], 
[0,1,2], 
[5,7,-2],
[-1,4,0]], dtype=float)
y = np.array([1, 1, 2, 3, 2, 3], dtype=int)  # Class labels: 1, 2, 3.
X_test = np.array([[1,-2,3]], dtype=float)
y_test = None  # Or np.array([1, 2]) if these are the supplied true labels.

order = 3  # 1 = linear; 2, 3, ... = polynomial.
lmd = 0.0  # 0 = ordinary regression; positive = ridge, if requested.


## 2. Construct the polynomial design matrix
For p raw input features and polynomial order n, including the offset:

$$
d = \binom{p+n}{n} = \frac{(p+n)!}{p!\,n!}.
$$

The training matrix P (`X_poly`) has shape **m by d**. At order 1, its columns are the constant and the original features. At higher orders it also contains powers and interactions. The fitted `poly` object must be reused for test samples.


In [30]:
X = np.array(X, dtype=float)
assert len(X.shape) == 2, "Use one sample per row; one feature uses [[a], [b], ...]."
assert lmd >= 0, "Lambda must be zero or positive."
poly = PolynomialFeatures(degree=order)
X_poly = poly.fit_transform(X)
raw_names = [f"x{i + 1}" for i in range(X.shape[1])]
terms = poly.get_feature_names_out(raw_names)
print("Polynomial terms:", terms)
print("Polynomial matrix shape:", X_poly.shape)


Polynomial terms: ['1' 'x1' 'x2' 'x3' 'x1^2' 'x1 x2' 'x1 x3' 'x2^2' 'x2 x3' 'x3^2' 'x1^3'
 'x1^2 x2' 'x1^2 x3' 'x1 x2^2' 'x1 x2 x3' 'x1 x3^2' 'x2^3' 'x2^2 x3'
 'x2 x3^2' 'x3^3']
Polynomial matrix shape: (6, 20)


## 3. One-hot encode the training labels
For three classes with class order `[1, 2, 3]`:

```text
Class 1 -> [1, 0, 0]
Class 2 -> [0, 1, 0]
Class 3 -> [0, 0, 1]
```

One-hot encoding gives a target matrix Y with **m rows and k class columns**. Regression learns a separate score for each class instead of treating label numbers as quantities. `ohe.categories_[0]` records which class corresponds to each column.


In [31]:
y = np.array(y)
assert len(y.shape) == 1 or (len(y.shape) == 2 and y.shape[1] == 1), "Use one class label per training sample."
y = y.reshape(-1, 1)
assert y.shape[0] == X.shape[0], "X and y must have the same number of samples."

ohe = OneHotEncoder(sparse_output=False)
Y = ohe.fit_transform(y)
classes = ohe.categories_[0]
print("One-hot column labels:", classes)
print("One-hot target matrix:\n", Y)


One-hot column labels: [1 2 3]
One-hot target matrix:
 [[1. 0. 0.]
 [1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]
 [0. 1. 0.]
 [0. 0. 1.]]


## 4. Train one weight column per class
The dimensions satisfy:

$$
\underbrace{\hat Y}_{m\times k}
= \underbrace{P}_{m\times d}\underbrace{W}_{d\times k}.
$$

For ordinary regression (`lmd = 0`), use the left inverse if P has full column rank, or the right inverse if P has full row rank. If neither exists, use Tutorial 4's pseudoinverse to obtain the minimum-norm least-squares weights.

For ridge (`lmd > 0`), the same formulas include a positive penalty:

$$
\boxed{W_{\mathrm{primal}}=(P^TP+\lambda I_d)^{-1}P^TY},\qquad
\boxed{W_{\mathrm{dual}}=P^T(PP^T+\lambda I_m)^{-1}Y}.
$$

The manual `if/else` below selects the smaller matrix to invert. At lambda zero these reduce to the ordinary left/right inverse formulas when the required rank condition holds. Positive lambda makes the penalized matrices invertible even with rank deficiency. The ridge convention from Tutorials 5-6 penalizes all coefficients, including the intercept, using an unnormalized sum of squared errors.


In [32]:
m, d = X_poly.shape
rank = np.linalg.matrix_rank(X_poly)

# MANUAL IF/ELSE: the same regression formulas, now with one-hot targets Y.
if lmd == 0 and rank < m and rank < d:
    # Tut4 fallback: neither ordinary inverse exists for this rank-deficient P.
    W = np.linalg.pinv(X_poly) @ Y
    method = "Pseudoinverse (minimum-norm least squares)"
elif m >= d:
    W = inv(X_poly.T @ X_poly + lmd * np.identity(d)) @ X_poly.T @ Y
    method = "Primal ridge" if lmd > 0 else "Left inverse"
else:
    W = X_poly.T @ inv(X_poly @ X_poly.T + lmd * np.identity(m)) @ Y
    method = "Dual ridge" if lmd > 0 else "Right inverse"

print("Method:", method)
print("Polynomial matrix shape/rank:", X_poly.shape, rank)
print("Y shape:", Y.shape, " W shape:", W.shape)


Method: Right inverse
Polynomial matrix shape/rank: (6, 20) 6
Y shape: (6, 3)  W shape: (20, 3)


## 5. Display weights by class
Each column of W belongs to one class, in the order shown by `classes`. Each row belongs to one displayed polynomial term. The term `1` is the offset coefficient.

Keep the full-precision W for prediction; round only the displayed coefficients.


In [33]:
for j in range(len(classes)):
    print(f"\nClass {classes[j]}:")
    for i in range(len(terms)):
        print(f"{terms[i]}: {W[i, j]:.6f}")



Class 1:
1: -0.011768
x1: 0.001938
x2: -0.007462
x3: -0.014166
x1^2: -0.000317
x1 x2: 0.006907
x1 x3: 0.011570
x2^2: -0.000692
x2 x3: 0.001156
x3^2: -0.017196
x1^3: -0.008366
x1^2 x2: 0.002364
x1^2 x3: 0.003648
x1 x2^2: 0.013108
x1 x2 x3: 0.030272
x1 x3^2: 0.024615
x2^3: 0.004289
x2^2 x3: 0.031765
x2 x3^2: 0.029541
x3^3: -0.008907

Class 2:
1: 0.174823
x1: 0.168201
x2: 0.170160
x3: -0.000982
x1^2: 0.155364
x1 x2: 0.141068
x1 x3: 0.001178
x2^2: 0.143154
x2 x3: 0.000404
x3^2: -0.014154
x1^3: 0.081636
x1^2 x2: 0.043117
x1^2 x3: 0.033953
x1 x2^2: -0.028774
x1 x2 x3: 0.037446
x1 x3^2: -0.030531
x2^3: -0.048750
x2^2 x3: 0.048247
x2 x3^2: -0.055324
x3^3: -0.032102

Class 3:
1: 0.044184
x1: -0.010798
x2: 0.031688
x3: 0.067588
x1^2: -0.007193
x1 x2: -0.024083
x1 x3: -0.040892
x2^2: 0.011807
x2 x3: 0.023664
x3^2: 0.095756
x1^3: 0.004751
x1^2 x2: -0.009348
x1^2 x3: -0.037279
x1 x2^2: -0.040087
x1 x2 x3: -0.090462
x1 x3^2: -0.111704
x2^3: -0.000922
x2^2 x3: -0.067942
x2 x3^2: -0.032830
x3^3: 0.07

## 6. Check training accuracy
For each row of scores, select the largest score and decode its column index to the original class label:

$$
\hat c_i = \operatorname{classes}\left[\arg\max_j (PW)_{ij}\right],\qquad
\mathrm{accuracy}=\frac{\text{number of correct labels}}{\text{number of samples}}.
$$

`argmax` gives an index, not necessarily a label. Mapping through `classes` supports nonconsecutive numeric labels and strings; do not blindly add 1. Training accuracy describes the fitted training samples; test accuracy checks held-out/new samples with known labels.


In [34]:
train_scores = X_poly @ W
train_labels = classes[np.argmax(train_scores, axis=1)]
train_accuracy = accuracy_score(y.ravel(), train_labels)
print("Training predictions:", train_labels)
print("Training true labels:", y.ravel())
print("Training accuracy:", train_accuracy)


Training predictions: [1 1 2 3 2 3]
Training true labels: [1 1 2 3 2 3]
Training accuracy: 1.0


## 7. Predict test classes

$$
\mathrm{scores}_{\mathrm{test}} = \phi(X_{\mathrm{test}})W,\qquad
\hat c_i = \operatorname{classes}\left[\arg\max_j\mathrm{scores}_{ij}\right].
$$

Use the existing `poly.transform` mapping. There is **one score per class** for each sample. The largest score determines the predicted class; multiclass decoding uses `argmax`. Scores may be negative and are **not probabilities**. A tie selects the first class in the stored column order.


In [35]:
X_test = np.array(X_test, dtype=float)
X_test_poly = poly.transform(X_test)
scores = X_test_poly @ W
predicted_classes = classes[np.argmax(scores, axis=1)]
print("Score column labels:", classes)
print("Test scores:\n", scores)
print("Predicted classes:", predicted_classes)

if y_test is not None:
    y_test = np.array(y_test)
    assert len(y_test.shape) == 1 or (len(y_test.shape) == 2 and y_test.shape[1] == 1), "Use one class label per test sample."
    test_accuracy = accuracy_score(y_test.ravel(), predicted_classes)
    print("Test accuracy:", test_accuracy)


Score column labels: [1 2 3]
Test scores:
 [[-0.51646251  0.90008188  2.00518001]]
Predicted classes: [3]
